# Lezione 3 — Pratica: istogrammi e grafici a barre

Risolvi gli esercizi usando `df_can` e la lista `years`. Ogni esercizio porta suggerimenti; se ti blocchi, apri la soluzione, ma prova prima a scrivere il tuo codice.

## Preparazione: librerie e dati

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline
mpl.style.use('ggplot')

df_can = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv')
df_can.set_index('Country', inplace=True)

years = list(map(str, range(1980, 2014)))

print('Dati pronti:', df_can.shape)

## Esercizio 1 — Istogramma di Grecia, Albania e Bulgaria (1980-2013)

Crea un istogramma della distribuzione di immigrati da **Grecia, Albania e Bulgaria** tra il 1980 e il 2013.

Requisiti:
- 15 bin
- trasparenza (`alpha`) di 0,35
- **senza** impilare (`stacked=False`)
- usa un colore diverso per ogni paese (opzionale, ma consigliato)

**Suggerimenti:**
- Ricorda di selezionare le righe con `.loc[[...], years]` e di **trasporre** il risultato con `.transpose()` affinché ogni paese diventi una colonna.
- `np.histogram(df_t, 15)` ti dà `bin_edges` da passare come `xticks`.
- Non dimenticare `plt.title`, `plt.xlabel` e `plt.ylabel`.

In [ ]:
# TODO: seleziona Grecia, Albania e Bulgaria da df_can (colonne = years) e trasponile
# TODO: calcola count, bin_edges con np.histogram(..., 15)
# TODO: crea il grafico con kind='hist', bins=15, alpha=0.35, xticks=bin_edges, stacked=False
# TODO: aggiungi titolo ed etichette degli assi
# TODO: plt.show()


<details>
<summary>💡 Vedi soluzione</summary>

```python
# selezioniamo i 3 paesi e trasponiamo affinché ogni paese sia una colonna
df_gab = df_can.loc[['Greece', 'Albania', 'Bulgaria'], years].transpose()

# calcoliamo bin e conteggi grezzi con np.histogram
count, bin_edges = np.histogram(df_gab, 15)

# creiamo il grafico dell'istogramma multi-serie
df_gab.plot(kind='hist',
            figsize=(10, 6),
            bins=15,
            alpha=0.35,                # trasparenza richiesta
            xticks=bin_edges,
            stacked=False,             # senza impilare
            color=['coral', 'darkslateblue', 'mediumseagreen'])

plt.title('Istogramma di immigrazione da Grecia, Albania e Bulgaria (1980-2013)')
plt.ylabel('Numero di anni')
plt.xlabel('Numero di immigrati')

plt.show()
```
</details>

## Esercizio 2 — Barre verticali: immigrazione dall'India anno per anno

Crea un grafico a **barre verticali** con gli immigrati totali dall'**India** anno per anno (1980-2013). Aggiungi un'annotazione con freccia che segnali l'anno in cui il valore è stato più alto.

**Suggerimenti:**
- Estrai la riga dell'India con `df_can.loc['India', years]`.
- `kind='bar'` disegna barre verticali; usa `rot=90` per ruotare le etichette dell'asse x.
- Per trovare la posizione del valore massimo dentro la serie puoi usare `.values.argmax()` (ti dà l'indice numerico, che coincide con la posizione sull'asse x quando crei il grafico con pandas).
- `plt.annotate('', xy=(x_punta, y_punta), xytext=(x_base, y_base), xycoords='data', arrowprops=dict(arrowstyle='->', color='blue', lw=2))` disegna la freccia; un secondo `plt.annotate` con il testo descrive l'evento.

In [ ]:
# TODO: estrai la serie dell'India (df_can.loc['India', years])
# TODO: crea il grafico con kind='bar', figsize=(10, 6), rot=90
# TODO: aggiungi xlabel, ylabel e title
# TODO: trova l'anno (posizione) e il valore dove l'immigrazione è stata massima
# TODO: usa plt.annotate per disegnare una freccia che punti a quel picco
# TODO: plt.show()


<details>
<summary>💡 Vedi soluzione</summary>

```python
df_india = df_can.loc['India', years]

df_india.plot(kind='bar', figsize=(10, 6), rot=90)

plt.xlabel('Anno')
plt.ylabel('Numero di immigrati')
plt.title('Immigrati indiani in Canada (1980-2013)')

# posizione (indice numerico) e valore del picco massimo
peak_index = df_india.values.argmax()
peak_value = df_india.values.max()

# la freccia punta da poco prima del picco verso il picco stesso
plt.annotate('',
             xy=(peak_index, peak_value),
             xytext=(peak_index - 4, peak_value - 5000),
             xycoords='data',
             arrowprops=dict(arrowstyle='->', connectionstyle='arc3', color='blue', lw=2))

plt.annotate(f'Massimo: {peak_value} nel {df_india.index[peak_index]}',
             xy=(peak_index - 4, peak_value - 4500),
             va='bottom',
             ha='left')

plt.show()
```
</details>

## Esercizio 3 — Barre orizzontali: i 5 paesi che hanno contribuito meno

Crea un grafico a **barre orizzontali** con i **5 paesi che hanno contribuito meno** al totale dell'immigrazione (colonna `Total`), e annota il valore esatto accanto a ogni barra usando un ciclo con `enumerate`.

**Suggerimenti:**
- Ordina `df_can` per `Total` in modo crescente e prendi i primi 5 con `.head(5)` (quelli che hanno contribuito meno restano all'inizio se l'ordine è crescente).
- `kind='barh'` disegna barre orizzontali.
- Scorri la serie con `for index, value in enumerate(serie):` — `index` è la posizione verticale della barra, `value` è il totale di quel paese.
- `plt.annotate(label, xy=(value + margine, index), color='black')` colloca il testo accanto alla punta della barra; regola il margine a occhio in base alla scala dei tuoi dati.

In [ ]:
# TODO: ordina df_can per 'Total' crescente e prendi i 5 paesi con meno immigrazione totale
# TODO: crea il grafico con kind='barh'
# TODO: aggiungi xlabel e title
# TODO: scorri la serie con enumerate() e usa plt.annotate per mettere il valore accanto a ogni barra
# TODO: plt.show()


<details>
<summary>💡 Vedi soluzione</summary>

```python
# passo 1: ordinare in modo crescente e tenere i 5 minori
df_can.sort_values(by='Total', ascending=True, inplace=True)
df_bottom5 = df_can['Total'].head(5)

# passo 2: creare il grafico a barre orizzontali
df_bottom5.plot(kind='barh', figsize=(10, 6), color='steelblue')
plt.xlabel('Numero di immigrati')
plt.title('I 5 paesi che hanno contribuito di meno all\'immigrazione in Canada (1980-2013)')

# passo 3: annotare il valore esatto accanto a ogni barra
for index, value in enumerate(df_bottom5):
    label = format(int(value), ',')
    plt.annotate(label, xy=(value + 0.5, index), va='center', color='black')

plt.show()
```
</details>

## Ricapitolazione

- Un istogramma multi-serie ha bisogno di dati **trasposti**: paesi/serie come colonne, anni come righe.
- `argmax()` su `.values` ti dà la posizione del massimo dentro una serie di pandas.
- `enumerate()` è il modo idiomatico di annotare ogni barra di un `barh` con il suo valore esatto.